# Lab: Can an AI weather model forecast your event?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/evanwellmeyer/teaching/blob/main/ai_weather_lab/neuralgcm_event_lab.ipynb)

Pick a weather event, watch it in ERA5, and find out whether NeuralGCM could have forecast it. NeuralGCM (Kochkov et al. 2024) combines a conventional dynamical core with neural networks trained on ERA5.

Before you start, choose `Runtime → Change runtime type → T4 GPU`, then run the four setup cells. The last one has a menu to pick the NeuralGCM model.

In [ ]:
#@title Setup: install packages (~1 min)
!pip install -q neuralgcm gcsfs cartopy
# If the next cell fails with a version error, uncomment this line, run it, then
# restart the session (Runtime -> Restart session) and continue from the next cell:
# !pip install -q "neuralgcm==1.2.2" "dinosaur==1.3.6"

In [ ]:
#@title Setup: imports and ERA5 data
import pickle
import time

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import gcsfs
import jax
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from dinosaur import horizontal_interpolation, spherical_harmonic, xarray_utils
from IPython.display import HTML, display
from matplotlib import animation, colors, ticker

import neuralgcm

print("JAX devices:", jax.devices())
if jax.default_backend() != "gpu":
    print("\n*** No GPU found. Use Runtime -> Change runtime type -> GPU, then rerun. ***")

plt.rcParams.update({"font.size": 11, "figure.dpi": 100, "animation.embed_limit": 100})
G = 9.80665   # gravity, m/s^2
RD = 287.05   # gas constant of dry air, J/(kg K)

# ERA5 reanalysis, hourly at 0.25 degrees (Google's public copy, no account needed)
ERA5 = xr.open_zarr("gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3",
                    chunks=None, storage_options=dict(token="anon"))


def pretty_units(units):
    return units.replace("**-1", "⁻¹").replace("**-2", "⁻²").replace("**2", "²")


def show_era5_variables(search=""):
    # A table of the ERA5 variables whose name or description contains `search`,
    # with the ones NeuralGCM can forecast first
    forecastable = set(model.input_variables) | set(SINGLE_LEVEL)
    rows = []
    for name in ERA5.data_vars:
        description = ERA5[name].attrs.get("long_name", "")
        if search.lower() in (name + " " + description).lower():
            rows.append({
                "variable": name,
                "description": description,
                "units": pretty_units(ERA5[name].attrs.get("units", "")),
                "pressure levels": "yes" if "level" in ERA5[name].dims else "",
                "NeuralGCM forecasts it": "yes" if name in forecastable else "",
            })
    table = pd.DataFrame(rows).sort_values(["NeuralGCM forecasts it", "variable"], ascending=[False, True])
    try:   # in Colab: a table you can sort and search
        from google.colab import data_table
        display(data_table.DataTable(table, include_index=False, num_rows_per_page=25))
    except ImportError:
        with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
            display(table.set_index("variable"))


def crop(ds, lats, lons):
    # Cut out a box: lats = (south, north), lons = (west, east), with negative longitudes to the west.
    # Longitudes in the result run continuously from west to east, even across 0 or 180 degrees.
    west, east = lons
    offset = (ds.longitude - west) % 360
    ds = ds.isel(longitude=np.flatnonzero(offset.values <= (east - west) % 360))
    ds = ds.assign_coords(longitude=west + (ds.longitude - west) % 360).sortby("longitude")
    inside = (ds.latitude.values >= lats[0]) & (ds.latitude.values <= lats[1])
    return ds.isel(latitude=np.flatnonzero(inside)).sortby("latitude")


def in_plot_units(field, variable, level):
    # Convert ERA5 units to friendlier ones, and make a label
    units = field.attrs.get("units", "")
    name = field.attrs.get("long_name", variable.replace("_", " "))
    if variable == "total_precipitation":
        field, units = field * 1000, "mm/h"   # m of water in the past hour -> mm/h
    elif units == "K":
        field, units = field - 273.15, "°C"
    elif units == "Pa":
        field, units = field / 100, "hPa"
    elif units == "m**2 s**-2":
        field, units, name = field / G, "m", name.replace("Geopotential", "Geopotential height")
    elif units == "kg kg**-1":
        field, units = field * 1000, "g/kg"
    label = f"{name} ({pretty_units(units)})"
    if level is not None:
        label = f"{level} hPa {label}"
    field = field.drop_vars("level", errors="ignore")
    return field.assign_coords(time=field.time.values.astype("datetime64[ns]")), label


def event_data(colors, lines):
    # What the maps need: `colors` and `lines` are (field, variable, level), with field still in ERA5 units
    colors_field, colors_label = in_plot_units(*colors)
    lines_field, lines_label = in_plot_units(*lines)
    data = xr.Dataset({"colors": colors_field, "lines": lines_field})
    return data.assign_attrs(colors=colors[1], colors_label=colors_label, lines=lines[1], lines_label=lines_label)


def check_variable(variable, level):
    if variable not in ERA5:
        raise ValueError(f"ERA5 has no variable {variable!r}. Run show_era5_variables() to see the names.")
    if "level" in ERA5[variable].dims and level is None:
        raise ValueError(f"{variable} is on pressure levels: give it a level, for example 500")


def load_era5(variable, level=None, lines="mean_sea_level_pressure", lines_level=None):
    # ERA5 in the box (LATS, LONS), every 6 hours from START to END: `variable` for the colors and `lines` for the
    # contour lines, each at a pressure level if it has levels
    times = np.arange(np.datetime64(START), np.datetime64(END) + np.timedelta64(1, "h"), np.timedelta64(6, "h"))
    tic = time.time()
    loaded = []
    for name, lev in [(variable, level), (lines, lines_level)]:
        check_variable(name, lev)
        field = ERA5[name].sel(time=times)
        if "level" in field.dims:
            print(f"Loading {name} at {lev} hPa; variables on pressure levels take a minute or two...")
            field = field.sel(level=lev)
        loaded.append((crop(field, LATS, LONS).compute(), name, lev))
    print(f"Loaded ERA5 in {time.time() - tic:.0f} s")
    return event_data(*loaded)

In [ ]:
#@title Setup: maps and animations
PC = ccrs.PlateCarree()


def when(t):
    return pd.Timestamp(t).strftime("%b %d %Y, %H UTC")


def box_projection():
    # Map projection centered on the box, and the box's extent in it
    width = (LONS[1] - LONS[0]) % 360
    center = LONS[0] + width / 2
    return ccrs.PlateCarree(central_longitude=center), [-width / 2, width / 2, LATS[0], LATS[1]]


def color_style(data):
    # Color map and color scale for the colors field, from its values over all times
    variable, field = data.attrs["colors"], data.colors
    if variable == "total_precipitation":
        return "turbo", colors.LogNorm(0.1, 30)
    lo, hi = np.nanpercentile(field, [1, 99])
    if "cloud" in variable:   # clouds look like a satellite image: white on black
        return "gray", colors.PowerNorm(0.5, vmin=0, vmax=hi)
    if lo < 0 < hi:   # values of both signs: red above zero, blue below
        top = max(-lo, hi)
        return "RdBu_r", colors.Normalize(-top, top)
    return "viridis", colors.Normalize(lo, hi)


def line_levels(data):
    # Contour levels for the lines field, the same at all times: every 4 hPa for sea-level pressure,
    # otherwise about 12 round numbers across its range
    if data.attrs["lines"] == "mean_sea_level_pressure":
        return np.arange(900, 1080, 4)
    lo, hi = np.nanpercentile(data.lines, [1, 99])
    return ticker.MaxNLocator(12).tick_values(lo, hi)


def lowest_pressure(pressure):
    # The lowest sea-level pressure on a map, with its latitude and longitude
    p = pressure.stack(point=("latitude", "longitude"))
    return p.isel(point=p.argmin("point"))


def draw_map(ax, data, cmap, norm, levels):
    # One map at one time: one field in color, another as labeled lines (and the lowest sea-level pressure)
    proj, extent = box_projection()
    dark = cmap == "gray"
    field = data.colors
    if data.attrs["colors"] == "total_precipitation":
        field = field.where(field >= 0.1)   # leave dry areas blank
        ax.add_feature(cfeature.OCEAN, color="#eef3f7")
        ax.add_feature(cfeature.LAND, color="#e3e3e3")
    ax.pcolormesh(field.longitude, field.latitude, field.transpose("latitude", "longitude"),
                  transform=PC, cmap=cmap, norm=norm, shading="auto")
    line_color = "white" if dark else "black"
    lines = data.lines.transpose("latitude", "longitude")
    contours = ax.contour(lines.longitude, lines.latitude, lines, levels=levels, colors=line_color,
                          linewidths=0.7, transform=PC)
    if data.attrs["lines"] == "mean_sea_level_pressure":   # mark the lowest pressure
        low = lowest_pressure(data.lines)
        ax.text(float(low.longitude), float(low.latitude), f"L {float(low):.0f}", transform=PC, color=line_color,
                ha="center", va="center", fontsize=11,
                bbox=dict(boxstyle="round", fc="black" if dark else "white", ec="none", alpha=0.6))
    else:
        ax.clabel(contours, fontsize=8, fmt="%g")
    ax.coastlines(color="gold" if dark else "0.2", linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, edgecolor="gold" if dark else "0.3", linewidth=0.5)
    ax.add_feature(cfeature.STATES, edgecolor="gold" if dark else "0.5", linewidth=0.3)
    ax.set_extent(extent, crs=proj)


def shared_times(runs):
    times = None
    for data in runs.values():
        t = pd.DatetimeIndex(data.time.values).as_unit("ns")
        times = t if times is None else times.intersection(t)
    return times


def animate(runs, cmap=None, save=None):
    # Animated maps, one panel per dataset in `runs` ({name: data}), at the times they share. Color scale and
    # contour levels are set by the first dataset. cmap = None picks a color map automatically.
    # save = "name.gif" also saves the animation (find it in the Files panel on the left).
    if cmap is not None and cmap not in plt.colormaps():
        raise ValueError(f"Unknown color map {cmap!r}. See https://matplotlib.org/stable/users/explain/colors/colormaps.html")
    runs = {name: crop(data, LATS, LONS) for name, data in runs.items()}
    times = shared_times(runs)
    if len(times) == 0:
        print("These datasets share no times:")
        for name, data in runs.items():
            print(f"  {name}: {when(data.time.values[0])} to {when(data.time.values[-1])}")
        return
    first = next(iter(runs.values()))
    auto_cmap, norm = color_style(first)
    cmap = cmap or auto_cmap
    levels = line_levels(first)
    proj, extent = box_projection()
    panel_height = 5.5 * (extent[3] - extent[2]) / (extent[1] - extent[0])
    fig, axes = plt.subplots(1, len(runs), figsize=(5.5 * len(runs), panel_height + 1.5),
                             subplot_kw={"projection": proj}, layout="constrained", squeeze=False)
    axes = axes[0]
    fig.colorbar(plt.cm.ScalarMappable(norm, cmap), ax=axes, orientation="horizontal", shrink=0.6,
                 label=first.attrs["colors_label"], format="%g")

    def frame(i):
        for ax, (name, data) in zip(axes, runs.items()):
            ax.clear()
            draw_map(ax, data.sel(time=times[i]), cmap, norm, levels)
            ax.set_title(name)
        fig.suptitle(f"{EVENT}, {when(times[i])}\nlines: {first.attrs['lines_label']}")

    print(f"Building the animation ({len(times)} frames)...")
    frame(0)
    anim = animation.FuncAnimation(fig, frame, frames=len(times), interval=400)
    if save:
        anim.save(save, writer="pillow", fps=3)
        print(f"Saved {save}")
    display(HTML(anim.to_jshtml(default_mode="once")))
    plt.close(fig)

In [ ]:
#@title Setup: load NeuralGCM (~1 min)
# Only the v1_precip model predicts rain. The 1.4 degree models are sharper but about 4 times slower.
MODEL_NAME = "v1_precip/stochastic_precip_2_8_deg.pkl"  #@param ["v1_precip/stochastic_precip_2_8_deg.pkl", "v1/deterministic_2_8_deg.pkl", "v1/deterministic_1_4_deg.pkl", "v1/stochastic_1_4_deg.pkl"]
OUTPUT_HOURS = 6   # forecasts are saved every 6 hours

gcs = gcsfs.GCSFileSystem(token="anon")
with gcs.open(f"gs://neuralgcm/models/{MODEL_NAME}", "rb") as f:
    model = neuralgcm.PressureLevelModel.from_checkpoint(pickle.load(f))

# For averaging ERA5 onto the model's grid
era5_grid = spherical_harmonic.Grid(
    latitude_nodes=ERA5.sizes["latitude"],
    longitude_nodes=ERA5.sizes["longitude"],
    latitude_spacing=xarray_utils.infer_latitude_spacing(ERA5.latitude),
    longitude_offset=xarray_utils.infer_longitude_offset(ERA5.longitude),
)
regridder = horizontal_interpolation.ConservativeRegridder(era5_grid, model.data_coords.horizontal, skipna=True)


def to_model_grid(ds):
    return xarray_utils.fill_nan_with_nearest(xarray_utils.regrid(ds, regridder))


surface = to_model_grid(ERA5[["geopotential_at_surface"]].sel(time=["2000-01-01T00"]).compute())
surface_geopotential = surface.geopotential_at_surface.isel(time=0, drop=True)


def load_initial_state(start):
    # The ERA5 state at `start`, on the model grid
    raw = (
        ERA5[model.input_variables + model.forcing_variables]
        # The model expects sea surface temperature and sea ice from 24 h earlier, as in training
        .pipe(xarray_utils.selective_temporal_shift,
              variables=model.forcing_variables, time_shift="24 hours")
        .sel(time=[np.datetime64(start)])
        .compute()
    )
    return to_model_grid(raw)


def distance_km(ds, lat, lon):
    # Great-circle distance from (lat, lon) to every grid point of ds
    lat_r, lon_r = np.deg2rad(ds.latitude), np.deg2rad(ds.longitude)
    cos_distance = (np.sin(lat_r) * np.sin(np.deg2rad(lat))
                    + np.cos(lat_r) * np.cos(np.deg2rad(lat)) * np.cos(lon_r - np.deg2rad(lon)))
    return 6371 * np.arccos(np.clip(cos_distance, -1, 1))


NUDGE_UNITS = {   # variables you can nudge: (the units of `amount`, factor from those units to the model's)
    "temperature": ("K", 1.0),
    "specific_humidity": ("g/kg", 1e-3),
    "u_component_of_wind": ("m/s", 1.0),
    "v_component_of_wind": ("m/s", 1.0),
    "geopotential": ("m of height", G),
}


def apply_nudge(state, variable, amount, lat, lon, radius_km=500, layer_hPa=(850, 300)):
    # Add `amount` to `variable` at (lat, lon), fading out over radius_km, between two pressure levels
    if variable not in NUDGE_UNITS:
        raise ValueError(f"You can nudge one of: {', '.join(NUDGE_UNITS)}")
    bottom, top = max(layer_hPa), min(layer_hPa)
    blob = np.exp(-(distance_km(state, lat, lon) / radius_km) ** 2)
    in_layer = (state.level <= bottom) & (state.level >= top)
    change = amount * NUDGE_UNITS[variable][1] * blob.where(in_layer, 0.0)
    nudged = state.copy()
    new = state[variable] + change
    if variable == "specific_humidity":
        new = new.clip(min=0)   # humidity cannot go below zero
    nudged[variable] = new.transpose(*state[variable].dims)
    return nudged


def forecast(start, end, nudge=None):
    # Run NeuralGCM from the ERA5 state at `start` until `end`, saving every 6 hours.
    # nudge = dict(variable=..., amount=..., lat=..., lon=..., ...) changes the starting state first (apply_nudge).
    tic = time.time()
    state = load_initial_state(start)
    if nudge is not None:
        state = apply_nudge(state, **nudge)
    print(f"Loaded the ERA5 starting state in {time.time() - tic:.0f} s")
    inputs = model.inputs_from_xarray(state.isel(time=0))
    forcings = model.forcings_from_xarray(state.isel(time=0))
    model_state = model.encode(inputs, forcings, jax.random.key(0))
    days = int(np.ceil((np.datetime64(end) - np.datetime64(start)) / np.timedelta64(1, "D")))
    print("Running one day at a time (the first day includes about a minute of compiling the model)...")
    pieces = []
    for day in range(days + 1):   # one day at a time, so JAX compiles the model only once
        model_state, out = model.unroll(
            model_state,
            model.forcings_from_xarray(state.head(time=1)),   # sea surface temperature and sea ice stay fixed
            steps=24 // OUTPUT_HOURS,
            timedelta=np.timedelta64(OUTPUT_HOURS, "h"),
            start_with_input=True,
        )
        pieces.append(jax.device_get(out))
        day_status(pieces[-1], start, day, days, tic)
    out = jax.tree.map(lambda *days: np.concatenate(days), *pieces)
    hours = np.arange(len(out["temperature"])) * OUTPUT_HOURS
    out = model.data_to_xarray(out, times=hours)
    out = out.assign_coords(time=np.datetime64(start, "ns") + hours * np.timedelta64(1, "h"))
    print(f"Forecast from {when(start)} done in {time.time() - tic:.0f} s")
    return out.sel(time=slice(None, np.datetime64(end, "ns")))


def day_status(piece, start, day, days, tic):
    # Print how far the forecast has got, and the lowest sea-level pressure in the box at the end of the day
    hours = (day * 24 + np.arange(24 // OUTPUT_HOURS) * OUTPUT_HOURS)
    one_day = model.data_to_xarray(piece, times=hours).isel(time=-1)
    lowest = float(crop(sea_level_pressure(one_day), LATS, LONS).min()) / 100
    reached = np.datetime64(start) + np.timedelta64(int(hours[-1]), "h")
    print(f"  day {day + 1} of {days + 1}: reached {when(reached)}, lowest pressure in your box "
          f"{lowest:.0f} hPa  ({time.time() - tic:.0f} s)", flush=True)


# ---- ERA5 variables computed from the model's pressure levels ----
def above_ground(out):
    # Same grid; copying the coordinates avoids tiny rounding differences between them
    surface = surface_geopotential.assign_coords(latitude=out.latitude, longitude=out.longitude)
    return out.geopotential > surface


def column(out, x):
    # Amount of x in the column above the ground, kg/m^2
    return x.where(above_ground(out), 0.0).integrate("level") * 100 / G


def near_ground(out, x):
    # x on the lowest pressure level that is above the ground
    return x.where(above_ground(out)).sortby("level", ascending=False).bfill("level").isel(level=0)


def sea_level_pressure(out):
    # Pa, from the 1000 hPa height and temperature (hypsometric equation)
    return 100000 * np.exp(out.geopotential.sel(level=1000) / (RD * out.temperature.sel(level=1000)))


def precipitation(out):
    # m of water per hour, averaged over the past 6 hours (ERA5 gives it for the past hour)
    total = out.precipitation_cumulative_mean.squeeze("surface", drop=True)   # m since the start
    return (total.diff("time") / OUTPUT_HOURS).reindex(time=out.time)


SINGLE_LEVEL = {
    "total_precipitation": precipitation,
    "mean_sea_level_pressure": sea_level_pressure,
    "total_column_water_vapour": lambda out: column(out, out.specific_humidity),
    "total_column_cloud_liquid_water": lambda out: column(out, out.specific_cloud_liquid_water_content),
    "total_column_cloud_ice_water": lambda out: column(out, out.specific_cloud_ice_water_content),
    "2m_temperature": lambda out: near_ground(out, out.temperature),
    "10m_u_component_of_wind": lambda out: near_ground(out, out.u_component_of_wind),
    "10m_v_component_of_wind": lambda out: near_ground(out, out.v_component_of_wind),
}


def model_variable(out, variable, level):
    # One ERA5 variable from the forecast, in ERA5's units
    if variable == "total_precipitation" and "precipitation_cumulative_mean" not in out:
        raise ValueError(f"{MODEL_NAME} does not predict rain. Pick another variable, or choose the v1_precip "
                         "model in the setup cell and run it and the forecast again.")
    if variable in SINGLE_LEVEL:
        field = SINGLE_LEVEL[variable](out)
    elif variable in out and "level" in out[variable].dims:
        if level is None:
            raise ValueError(f"{variable} is on pressure levels: give it a level, for example 500")
        field = out[variable].sel(level=level)
    else:
        on_levels = [name for name in out.data_vars if "level" in out[name].dims]
        raise ValueError(f"NeuralGCM does not predict {variable}. It predicts these on pressure levels: "
                         f"{', '.join(on_levels)}; and these single-level ones: {', '.join(SINGLE_LEVEL)}")
    return field.assign_attrs(ERA5[variable].attrs)


def model_field(out, variable, level=None, lines="mean_sea_level_pressure", lines_level=None):
    # The forecast, ready to animate next to load_era5 with the same arguments
    return event_data((model_variable(out, variable, level), variable, level),
                      (model_variable(out, lines, lines_level), lines, lines_level))


def spread_region(nudge):
    # A lat-lon box that holds both the nudge and the event box, with a margin
    width = (LONS[1] - LONS[0]) % 360
    to_star = (nudge["lon"] - LONS[0]) % 360   # degrees east of the box's west edge
    if to_star <= width:
        west, east = LONS[0], LONS[0] + width
    elif to_star < 180 + width / 2:            # the star is closer going east
        west, east = LONS[0], LONS[0] + to_star
    else:                                      # going west
        west, east = LONS[0] - (360 - to_star), LONS[0] + width
    south = max(min(LATS[0], nudge["lat"]) - 15, -85)
    north = min(max(LATS[1], nudge["lat"]) + 15, 85)
    return (south, north), (west - 15, east + 15)


def contour_levels(field, variable):
    # The same contour levels at all times: every 60 m for geopotential height, 4 hPa for sea-level pressure,
    # otherwise about 12 round numbers across the range
    lo, hi = np.nanpercentile(field, [1, 99])
    step = {"geopotential": 60, "mean_sea_level_pressure": 4}.get(variable)
    if step:
        return np.arange(np.floor(lo / step) * step, hi + step, step)
    return ticker.MaxNLocator(12).tick_values(lo, hi)


def show_spread(nudged, control, nudge, variable="geopotential", level=500, save=None):
    # Animated map of one variable as lines, without the nudge (thick gray) and with it (thin red),
    # over a region that holds the nudge (star) and your event (box)
    if not np.array_equal(nudged.time.values, control.time.values):
        raise ValueError("The nudged forecast and the control (fc) must have the same start and end. "
                         "Rerun both with the current FORECAST_START and END.")

    # A check: after 6 hours, the nudge cannot have changed anything far away by much
    slp_diff = (sea_level_pressure(nudged) - sea_level_pressure(control)) / 100
    far_away = distance_km(slp_diff, nudge["lat"], nudge["lon"]) > 3000
    early = float(np.abs(slp_diff.isel(time=1).where(far_away)).max())
    print(f"Largest sea-level pressure difference more than 3000 km from the star after 6 hours: {early:.2g} hPa")
    if early > 0.5:
        print("That is too big to come from the nudge: the two runs differ for some other reason, "
              "so this map is not meaningful.")

    lats, lons = spread_region(nudge)
    runs = {}
    for name, out in [("without the nudge", control), ("with the nudge", nudged)]:
        runs[name], label = in_plot_units(model_variable(out, variable, level), variable, level)
        runs[name] = crop(runs[name], lats, lons).transpose("time", "latitude", "longitude")
    without, with_nudge = runs.values()
    in_box = np.abs(crop(with_nudge - without, LATS, LONS)).max(("latitude", "longitude"))
    levels = contour_levels(without, variable)
    units = label[label.rfind("(") + 1:-1]

    width = lons[1] - lons[0]
    proj = ccrs.PlateCarree(central_longitude=lons[0] + width / 2)
    fig, ax = plt.subplots(figsize=(11, 11 * (lats[1] - lats[0]) / width + 1),
                           subplot_kw={"projection": proj}, layout="constrained")

    def frame(i):
        ax.clear()
        ax.add_feature(cfeature.OCEAN, color="#eef3f7")
        ax.add_feature(cfeature.LAND, color="#e3e3e3")
        ax.coastlines(color="0.4", linewidth=0.6)
        for f, color, lw in [(without, "0.35", 2.5), (with_nudge, "red", 1.0)]:
            ax.contour(f.longitude, f.latitude, f.isel(time=i), levels=levels, colors=color, linewidths=lw,
                       transform=PC)
        west, east = LONS[0], LONS[0] + (LONS[1] - LONS[0]) % 360
        ax.plot([west, east, east, west, west], [LATS[0], LATS[0], LATS[1], LATS[1], LATS[0]],
                color="black", lw=2, transform=PC)
        ax.plot(nudge["lon"], nudge["lat"], "*", color="gold", ms=20, mec="black", transform=PC)
        ax.set_extent([-width / 2, width / 2, lats[0], lats[1]], crs=proj)
        ax.legend(handles=[plt.Line2D([], [], color="0.35", lw=2.5, label="without the nudge"),
                           plt.Line2D([], [], color="red", lw=1, label="with the nudge"),
                           plt.Line2D([], [], color="gold", marker="*", ms=14, mec="black", ls="", label="the nudge"),
                           plt.Line2D([], [], color="black", lw=2, label=EVENT)],
                  loc="lower left", fontsize=9)
        ax.set_title(f"{label}, {when(without.time.values[i])}\n"
                     f"largest difference inside your box: {float(in_box[i]):.2g} {units}")

    print(f"Building the animation ({without.sizes['time']} frames)...")
    frame(0)
    anim = animation.FuncAnimation(fig, frame, frames=without.sizes["time"], interval=400)
    if save:
        anim.save(save, writer="pillow", fps=3)
        print(f"Saved {save}")
    display(HTML(anim.to_jshtml(default_mode="once")))
    plt.close(fig)


grid = model.data_coords.horizontal
print(f"Loaded {MODEL_NAME} ({grid.longitude_nodes} x {grid.latitude_nodes} grid points)")

---
## 1. Your event

Type in your event's name, a time window (UTC, on 00, 06, 12 or 18 h) and a box around it. Longitudes west of Greenwich are negative. ERA5 goes back to 1940.

In [ ]:
EVENT = "Hurricane Sandy"
START = "2012-10-25T00"
END = "2012-10-31T00"
LATS = (20, 50)     # south, north
LONS = (-85, -55)   # west, east

# Other events to try:
# Hurricane Helene:               START = "2024-09-24T00", END = "2024-09-28T00", LATS = (15, 45), LONS = (-95, -70)
# Typhoon Tip:                    START = "1979-10-08T00", END = "1979-10-16T00", LATS = (5, 40),  LONS = (120, 160)
# 1993 Storm of the Century:      START = "1993-03-11T00", END = "1993-03-15T00", LATS = (20, 55), LONS = (-100, -60)
# Texas cold-air outbreak:        START = "2021-02-10T00", END = "2021-02-18T00", LATS = (20, 55), LONS = (-115, -80)
# Winter Storm Elliott:           START = "2022-12-21T00", END = "2022-12-27T00", LATS = (25, 60), LONS = (-110, -60)
# Storm Eunice (Europe):          START = "2022-02-16T00", END = "2022-02-20T00", LATS = (40, 65), LONS = (-30, 20)
# Pacific Northwest heat dome:    START = "2021-06-24T00", END = "2021-07-02T00", LATS = (35, 60), LONS = (-135, -105)
# California atmospheric rivers:  START = "2023-01-01T00", END = "2023-01-11T00", LATS = (25, 50), LONS = (-150, -110)

Now design your map from the ERA5 variables in the table below: one variable in color, another as lines, and a [color map](https://matplotlib.org/stable/users/explain/colors/colormaps.html). Some variables, such as `temperature` and `geopotential`, are stored on 37 pressure levels from 1000 hPa (near the ground) up to 1 hPa; for those, also choose the level, for example 850 hPa (about 1.5 km up) or 500 hPa (about 5.5 km). For single-level variables, such as `total_precipitation`, the level is `None`. To compare with NeuralGCM later, pick variables it forecasts. With sea-level pressure as the lines, they are every 4 hPa and the lowest pressure is marked L.

In [ ]:
show_era5_variables()

In [ ]:
COLORS = "total_column_water_vapour"   # the variable shown in color
COLORS_LEVEL = None                    # only for variables on pressure levels: which level, in hPa (e.g. 850)
LINES = "mean_sea_level_pressure"      # the variable shown as lines (e.g. "geopotential" with LINES_LEVEL = 500)
LINES_LEVEL = None
CMAP = None                            # the color map; None picks one

era5 = load_era5(COLORS, COLORS_LEVEL, LINES, LINES_LEVEL)
animate({"ERA5": era5}, cmap=CMAP)

---
## 2. Forecast it

Start NeuralGCM from the ERA5 state a few days before your event.

In [ ]:
FORECAST_START = "2012-10-24T00"
fc = forecast(FORECAST_START, END)

Design the comparison the same way. NeuralGCM predicts temperature, geopotential, wind, humidity and cloud water on pressure levels, and rain. From these we also get sea-level pressure, column water vapor and cloud water, and approximate 2 m temperature and 10 m wind.

In [ ]:
COLORS = "total_precipitation"   # the variable shown in color
COLORS_LEVEL = None              # only for variables on pressure levels: which level, in hPa (e.g. 850)
LINES = "mean_sea_level_pressure"
LINES_LEVEL = None
CMAP = None

maps = {
    "ERA5": load_era5(COLORS, COLORS_LEVEL, LINES, LINES_LEVEL),
    "NeuralGCM": model_field(fc, COLORS, COLORS_LEVEL, LINES, LINES_LEVEL),
}
animate(maps, cmap=CMAP)

---
## a small change far away

Nudge the starting state a little, somewhere far from your event, and run the same forecast again. You can warm or cool the air, add or remove moisture, push the wind, or raise or lower the pressure surfaces.

In [ ]:
NUDGE = dict(
    variable="temperature",   # or "specific_humidity" (g/kg), "u_component_of_wind" or
                              # "v_component_of_wind" (m/s), "geopotential" (m of height)
    amount=1.0,               # how much, here in K (negative values work too)
    lat=40, lon=160,          # where
    radius_km=500,            # how wide
    layer_hPa=(850, 300),     # from which pressure level up to which
)
nudged = forecast(FORECAST_START, END, nudge=NUDGE)

Watch the two forecasts as lines: gray without the nudge, red with it. Where they agree, the red sits on top of the gray. Choose what to draw; 500 hPa geopotential height shows the large-scale waves well.

In [ ]:
LINES = "geopotential"
LINES_LEVEL = 500

show_spread(nudged, fc, NUDGE, LINES, LINES_LEVEL)

---
## your event in 3D

Draw a surface around everything above a threshold, in ERA5 and in your forecast, and move through time with the slider. Drag to rotate, scroll to zoom. Use any variable on pressure levels, or `"cloud_water"` (liquid plus ice) or `"wind_speed"`. Run the setup cell first. Loading ERA5 in 3D takes a few minutes; fewer times (a larger `EVERY_HOURS`) load faster.

In [ ]:
#@title Setup: 3D views
import plotly.graph_objects as go
import shapely
import shapely.affinity
from plotly.subplots import make_subplots

LEVELS_3D = [1000, 975, 950, 925, 900, 875, 850, 825, 800, 775, 750, 700, 650, 600, 550, 500, 450, 400,
             350, 300, 250, 225, 200, 175, 150, 125, 100]   # hPa, from the ground to about 16 km
COMBINED = {   # 3D variables made from two ERA5 variables
    "cloud_water": ["specific_cloud_liquid_water_content", "specific_cloud_ice_water_content"],
    "wind_speed": ["u_component_of_wind", "v_component_of_wind"],
}


def field_3d(source, variable, times):
    # One variable on pressure levels at several times, in the box, from ERA5 (source="ERA5") or a forecast
    parts = []
    for name in COMBINED.get(variable, [variable]):
        if source == "ERA5":
            if name not in ERA5 or "level" not in ERA5[name].dims:
                raise ValueError(f"{name} is not an ERA5 variable on pressure levels; see show_era5_variables()")
            da = crop(ERA5[name].sel(time=times, level=LEVELS_3D), LATS, LONS)
            k = int(np.ceil(da.sizes["longitude"] / 40))   # fewer points keep the 3D view quick
            da = da.compute().coarsen(latitude=k, longitude=k, boundary="trim").mean()
        else:
            if name not in source:
                raise ValueError(f"NeuralGCM does not predict {name}")
            da = crop(source[name].sel(time=times, level=LEVELS_3D), LATS, LONS)
        parts.append(da.assign_attrs(ERA5[name].attrs))
    if variable == "cloud_water":
        field = (parts[0] + parts[1]).assign_attrs(parts[0].attrs, long_name="Cloud water and ice")
    elif variable == "wind_speed":
        field = np.hypot(parts[0], parts[1]).assign_attrs(units="m s**-1", long_name="Wind speed")
    else:
        field = parts[0]
    levels = field.level
    field, label = in_plot_units(field, variable, None)
    return field.assign_coords(level=levels).transpose("time", "level", "latitude", "longitude"), label


def coastline_lines():
    # Coastlines inside the box as x, y lists (separated by None), for drawing on the floor of the 3D view
    west, (south, north) = LONS[0], LATS
    width = (LONS[1] - LONS[0]) % 360
    xs, ys = [], []
    for geometry in cfeature.COASTLINE.with_scale("50m").geometries():
        for shift in (-360, 0, 360):   # catch boxes that cross 180 degrees
            moved = shapely.affinity.translate(geometry, xoff=shift)
            clipped = moved.intersection(shapely.box(west, south, west + width, north))
            for line in getattr(clipped, "geoms", [clipped]):
                if not line.is_empty and line.geom_type == "LineString":
                    x, y = line.xy
                    xs += list(x) + [None]
                    ys += list(y) + [None]
    return xs, ys


def show_3d(variable, threshold, fc=None, every_hours=12, save=None):
    # 3D view of the box, with a time slider: a surface around everything above `threshold`, from ERA5 and (if fc
    # is given) the forecast, side by side. Drag to rotate, scroll to zoom. save = "name.html" also saves it.
    times = np.arange(np.datetime64(START, "ns"), np.datetime64(END, "ns") + np.timedelta64(1, "h"),
                      np.timedelta64(every_hours, "h"))
    sources = {"ERA5": "ERA5"}
    if fc is not None:
        times = times[np.isin(times, fc.time.values)]
        sources["NeuralGCM"] = fc
    if len(times) == 0:
        print("The forecast does not cover START to END. Check FORECAST_START.")
        return
    print(f"Loading {len(times)} times in 3D (ERA5 on pressure levels takes a while)...")
    tic = time.time()
    fields = {}
    for name, source in sources.items():
        fields[name], label = field_3d(source, variable, times)
    print(f"Loaded in {time.time() - tic:.0f} s")
    top = max(float(f.max()) for f in fields.values())
    if threshold >= top:
        print(f"Nothing is above {threshold:g}: the largest value is {top:.3g}. Try a smaller THRESHOLD.")
        return

    fig = make_subplots(rows=1, cols=len(fields), specs=[[{"type": "scene"}] * len(fields)],
                        subplot_titles=list(fields), horizontal_spacing=0.02)
    coast_x, coast_y = coastline_lines()
    surfaces = []   # trace numbers of the surfaces, which change with time
    for col, f in enumerate(fields.values(), start=1):
        height_km = 7.0 * np.log(1000.0 / f.level.values)   # approximate height of each pressure level
        z, y, x = np.meshgrid(height_km, f.latitude.values, f.longitude.values, indexing="ij")
        surfaces.append(len(fig.data))
        fig.add_trace(go.Isosurface(
            x=x.ravel(), y=y.ravel(), z=z.ravel(), value=f.isel(time=0).values.ravel(),
            isomin=threshold, isomax=top, surface_count=3, opacity=0.5, colorscale="Viridis",
            caps=dict(x_show=False, y_show=False, z_show=False),
            colorbar=dict(title=label, len=0.6), showscale=(col == 1),
        ), row=1, col=col)
        fig.add_trace(go.Scatter3d(x=coast_x, y=coast_y, z=[0] * len(coast_x), mode="lines",
                                   line=dict(color="black", width=3), showlegend=False), row=1, col=col)

    # One frame per time; each frame only replaces the values of the surfaces
    names = [when(t) for t in times]
    fig.frames = [go.Frame(name=names[i], traces=surfaces,
                           data=[go.Isosurface(value=f.isel(time=i).values.ravel()) for f in fields.values()])
                  for i in range(len(times))]
    fig.update_layout(sliders=[dict(
        currentvalue=dict(prefix="time: "), pad=dict(t=30),
        steps=[dict(label=name, method="animate",
                    args=[[name], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
               for name in names],
    )])

    width = (LONS[1] - LONS[0]) % 360
    scene = dict(xaxis_title="longitude", yaxis_title="latitude", zaxis_title="height (km)",
                 aspectmode="manual", aspectratio=dict(x=1, y=(LATS[1] - LATS[0]) / width, z=0.5))
    fig.update_layout(height=650, title=f"{EVENT}: {label} above {threshold:g}",
                      **{f"scene{'' if i == 1 else i}": scene for i in range(1, len(fields) + 1)})
    if save:
        fig.write_html(save)
        print(f"Saved {save}")
    fig.show()

In [ ]:
VARIABLE_3D = "cloud_water"   # e.g. "wind_speed" with THRESHOLD = 30 shows the jet stream
THRESHOLD = 0.05              # in the plot's units: here g/kg
EVERY_HOURS = 12              # time between slider steps (6, 12 or 24)

show_3d(VARIABLE_3D, THRESHOLD, fc, every_hours=EVERY_HOURS)
# show_3d(VARIABLE_3D, THRESHOLD, fc, every_hours=EVERY_HOURS, save="my_event_3d.html")   # to save it